# Model Merging & Quantization

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/fine-tuning-alignment/05-model-merging-and-quantization

A from-scratch NumPy walk through (1) merging two fine-tunes via weight averaging and task-vector arithmetic on toy weights, and (2) uniform post-training quantization at varying bit-widths. No API keys, no network.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(0)

## Part 1 — Task vectors and merging

We model a 1024-dim weight vector. Pretend the **base** model is θ_base, and we have two fine-tunes:

- θ_math = θ_base + small bump along a 'math' direction
- θ_chat = θ_base + small bump along a 'chat' direction

The task vectors are τ_math = θ_math − θ_base and τ_chat = θ_chat − θ_base. Merging is arithmetic on these vectors.

In [ ]:
D = 1024
theta_base = rng.normal(0, 1, D)

# Hidden 'skill directions' the fine-tunes push toward.
dir_math = rng.normal(0, 1, D); dir_math /= np.linalg.norm(dir_math)
dir_chat = rng.normal(0, 1, D); dir_chat /= np.linalg.norm(dir_chat)

theta_math = theta_base + 0.3 * dir_math
theta_chat = theta_base + 0.3 * dir_chat

tau_math = theta_math - theta_base
tau_chat = theta_chat - theta_base

# A 'skill score' = projection of the model onto a skill direction.
def score(theta, direction):
    return float((theta - theta_base) @ direction)

print(f'math score(θ_math)  = {score(theta_math, dir_math):.3f}')
print(f'chat score(θ_chat)  = {score(theta_chat, dir_chat):.3f}')
print(f'math score(θ_chat)  = {score(theta_chat, dir_math):.3f}  (≈0: chat fine-tune has no math)')

### Compare merging strategies

- **Average**: (θ_math + θ_chat) / 2
- **Task-vector add**: θ_base + τ_math + τ_chat
- **SLERP**: spherical interpolation between θ_math and θ_chat at t = 0.5

In [ ]:
theta_avg = 0.5 * (theta_math + theta_chat)
theta_addvec = theta_base + tau_math + tau_chat

def slerp(a, b, t):
    a_n = a / np.linalg.norm(a)
    b_n = b / np.linalg.norm(b)
    omega = np.arccos(np.clip(a_n @ b_n, -1, 1))
    if omega < 1e-6:
        return (1 - t) * a + t * b
    s = np.sin(omega)
    return (np.sin((1 - t) * omega) / s) * a + (np.sin(t * omega) / s) * b

theta_slerp = slerp(theta_math, theta_chat, 0.5)

for name, th in [('θ_math (parent)', theta_math),
                  ('θ_chat (parent)', theta_chat),
                  ('average merge', theta_avg),
                  ('task-vector add', theta_addvec),
                  ('SLERP (t=0.5)', theta_slerp)]:
    print(f'{name:22s}  math={score(th, dir_math):+.3f}   chat={score(th, dir_chat):+.3f}')

Notice the task-vector-add model gets *both* skills at full strength; plain averaging halves each (it's literally the mean of the two task vectors). SLERP sits between — geometry-aware but not additive.

### Task-vector negation

Now assume θ_chat is actually a **toxic** fine-tune. We want math skill *plus* explicit removal of toxic behaviour: θ_base + τ_math − τ_chat.

In [ ]:
theta_safe = theta_base + tau_math - tau_chat
print(f'safe-merge math = {score(theta_safe, dir_math):+.3f}   chat = {score(theta_safe, dir_chat):+.3f}')
print('chat score is negative — model is pushed *away* from that direction.')

## Part 2 — Uniform post-training quantization

We quantize a single FP32 weight tensor to `b` bits via uniform symmetric scaling, then measure the quantization noise. Plot accuracy proxy vs bit-width.

In [ ]:
def quantize(w, bits):
    if bits >= 32:
        return w.copy()
    vmax = np.abs(w).max()
    levels = 2 ** bits
    step = 2 * vmax / (levels - 1)
    q_idx = np.round((w + vmax) / step)
    return q_idx * step - vmax

W = rng.normal(0, 1, size=(256, 256))
bits_grid = [1, 2, 4, 8, 16]
errs = []
for b in bits_grid:
    Wq = quantize(W, b)
    err = np.linalg.norm(W - Wq) / np.linalg.norm(W)
    errs.append(err)
    print(f'{b:2d}-bit: noise = {err*100:6.2f}%   levels = {2**b}')

fig, ax = plt.subplots()
ax.semilogy(bits_grid, errs, marker='o', color='#6366f1')
ax.set_xlabel('bit-width')
ax.set_ylabel('relative quantization noise (log)')
ax.set_title('Uniform PTQ: noise drops geometrically with bits')
ax.invert_xaxis()  # fewer bits → harder
plt.show()

Noise roughly halves with every extra bit (the noise floor scales like 2^{-b}). At INT8 it's already a fraction of a percent; below INT4 it explodes.

## ✏️ Your turn

Implement `compute_memory_gb(num_params, bits)` that returns the memory cost in GB for a model of `num_params` parameters stored at `bits`-bit precision. Verify that a 7B model at INT4 takes around 3.3 GB.

In [ ]:
def compute_memory_gb(num_params, bits):
    # TODO(you): bits/8 bytes per parameter, then convert to GB (divide by 1024**3).
    return 0.0

mem_int4 = compute_memory_gb(7_000_000_000, 4)
assert 3.0 < mem_int4 < 3.7, f'7B INT4 should be ~3.3 GB, got {mem_int4}'
print(f'7B INT4 ≈ {mem_int4:.2f} GB  ✓')
print(f'7B FP16  ≈ {compute_memory_gb(7_000_000_000, 16):.2f} GB')
print(f'7B FP32  ≈ {compute_memory_gb(7_000_000_000, 32):.2f} GB')

<details><summary>Solution</summary>

```python
def compute_memory_gb(num_params, bits):
    bytes_per_param = bits / 8
    return num_params * bytes_per_param / (1024 ** 3)
```

At INT4: 7e9 × 0.5 / 1024³ ≈ 3.26 GB.
At FP16: 7e9 × 2   / 1024³ ≈ 13.04 GB.
At FP32: 7e9 × 4   / 1024³ ≈ 26.08 GB.

</details>